# Topic 03: Cleaning, transforming, and summarizing data

This notebook combines the course-provided lesson material and its exercise section. The answers follow the original prompts and use the pandas methods demonstrated in the lessons. Data files for this topic are stored in the adjacent `Datasets` folder. The requested conda environment is `CISC7204_BonusAssgn_ENV` (Python 3.13).

In [1]:
from pathlib import Path

DATA_DIR = Path("Datasets")

---

## Course lesson: 03_Pandas_DataFrames_Continued_01.ipynb

# Intro to Pandas
by Ryan Orsinger

## Module 3: DataFrames Continued

### Pandas DataFrames Continued - Identifying Missing Values
- Identifying and counting missing values
- Removing rows with missing information
- Dropping columns from a DataFrame

In [2]:
import pandas as pd

In [3]:
# Let's generate some data with missing values. 
# Real world data often has missing values
df = pd.DataFrame([
    {
        "item": "crackers",
        "serving_size": "4 crackers",
        "calories": 10,
        "fat": "1.1g",
        "sodium": "125mg",
        "price": 2.99,
        "discount": None
    },
    {
        "item": "club soda",
        "serving_size": "8 oz",
        "calories": None,
        "fat": None,
        "sodium": "75mg",
        "price": 2.25,
        "discount": None

    },
    {
        "item": "apple",
        "serving_size": 2,
        "calories": 95,
        "fat": None,
        "sodium": None,
        "price": 1.99,
        "discount": None
    },
    {
        "item": "banana",
        "serving_size": 3,
        "calories": 105,
        "fat": "0.4g",
        "sodium": "1mg",
        "price": None,
        "discount": None
    },
    {
        "item": "spam",
        "serving_size": "1 tin",
        "calories": None,
        "fat": None,
        "sodium": None,
        "price": None,
        "discount": None
    }
])

# Set the index to be the item name
df.set_index("item", inplace=True)
df

,serving_size,calories,fat,sodium,price,discount
item,,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99,None
club soda,8 oz,NaN,NaN,75mg,2.25,None
apple,2,95.0,NaN,NaN,1.99,None
banana,3,105.0,0.4g,1mg,NaN,None
spam,1 tin,NaN,NaN,NaN,NaN,None


In [4]:
# The .info method outputs data types and non-null value count
df.info()

<class 'pandas.DataFrame'>
Index: 5 entries, crackers to spam
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   serving_size  5 non-null      object 
 1   calories      3 non-null      float64
 2   fat           2 non-null      str    
 3   sodium        3 non-null      str    
 4   price         3 non-null      float64
 5   discount      0 non-null      object 
dtypes: float64(2), object(2), str(2)
memory usage: 280.0+ bytes


In [5]:
# Notice that missing values in a numeric column show as NaN, which means "not a number"
# For more on NaN, see https://en.wikipedia.org/wiki/NaN
df.calories

item
crackers      10.0
club soda      NaN
apple         95.0
banana       105.0
spam           NaN
Name: calories, dtype: float64

In [6]:
# NaN exists to allow us to do math without getting execution errors
# Many math functions ignore NaNs
df.calories.mean()

np.float64(70.0)

In [7]:
# By default, .value_counts ignores NaNs, too
df.sodium.value_counts()

sodium
125mg    1
75mg     1
1mg      1
Name: count, dtype: int64

In [8]:
# Use dropna=False to count missing values
df.sodium.value_counts(dropna=False)

sodium
NaN      2
125mg    1
75mg     1
1mg      1
Name: count, dtype: int64

In [9]:
# Notice that missing values in a string/object column show as None
df.fat

item
crackers     1.1g
club soda     NaN
apple         NaN
banana       0.4g
spam          NaN
Name: fat, dtype: str

In [10]:
# .isna() can operate on a column, returning a boolean series
df.sodium.isna()

item
crackers     False
club soda    False
apple         True
banana       False
spam          True
Name: sodium, dtype: bool

In [11]:
# .isna() can also operate on the entire dataframe
df.isna()

,serving_size,calories,fat,sodium,price,discount
item,,,,,,
crackers,False,False,False,False,False,True
club soda,False,True,True,False,False,True
apple,False,False,True,True,False,True
banana,False,False,False,False,True,True
spam,False,True,True,True,True,True


In [12]:
# Counting the number of nulls by column
print("Number of nulls by column")
df.isna().sum()

Number of nulls by column


serving_size    0
calories        2
fat             3
sodium          2
price           2
discount        5
dtype: int64

In [13]:
print("Proportion of nulls by column")
df.isna().mean()

Proportion of nulls by column


serving_size    0.0
calories        0.4
fat             0.6
sodium          0.4
price           0.4
discount        1.0
dtype: float64

In [14]:
# Counting the number of nulls by row
# Recall that .sum can run on columns or by row, by row with axis=1
print("Number of nulls by row")
df.isna().sum(axis=1)

Number of nulls by row


item
crackers     1
club soda    3
apple        3
banana       2
spam         5
dtype: int64

In [15]:
# Proportion of the number of nulls by row
# Recall that .sum can run on columns or by row, by row with axis=1
print("Proportion of nulls by row")
df.isna().mean(axis=1)

Proportion of nulls by row


item
crackers     0.166667
club soda    0.500000
apple        0.500000
banana       0.333333
spam         0.833333
dtype: float64

### Handling Missing Values
- There's no one right answer for all cases. 
- "It depends" is a common answer in data science. Context matters.
- Sometimes missing values might mean zero, depending on the context, so we can fill in zero.
- Sometimes, dropping entire rows or columns is appropriate
- Sometimes, filling missing values makes sense to keep the rest of the row or column's data

In [16]:
# Example of removing null values 
# dropna drops every row with a null value
# Since there is missing data in every row, this is quite destructive...
# the default axis argument is axis=0, which means row-wise
df.dropna()

,serving_size,calories,fat,sodium,price,discount
item,,,,,,


In [17]:
# dropna(axis=1) drops all columns with any missing values
# This is also too destructive to be helpful
df.dropna(axis=1)

,serving_size
item,
crackers,4 crackers
club soda,8 oz
apple,2
banana,3
spam,1 tin


In [18]:
# Let's review the dataframe
df

,serving_size,calories,fat,sodium,price,discount
item,,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99,None
club soda,8 oz,NaN,NaN,75mg,2.25,None
apple,2,95.0,NaN,NaN,1.99,None
banana,3,105.0,0.4g,1mg,NaN,None
spam,1 tin,NaN,NaN,NaN,NaN,None


In [19]:
# The discount column is adding no information here, so we can drop it
df.drop(columns="discount", inplace=True)
df

,serving_size,calories,fat,sodium,price
item,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99
club soda,8 oz,NaN,NaN,75mg,2.25
apple,2,95.0,NaN,NaN,1.99
banana,3,105.0,0.4g,1mg,NaN
spam,1 tin,NaN,NaN,NaN,NaN


In [20]:
# Reassign the df
# df.drop(index=["spam"], inplace=True) would produce the same result
df = df[df.index != "spam"]
df

,serving_size,calories,fat,sodium,price
item,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99
club soda,8 oz,NaN,NaN,75mg,2.25
apple,2,95.0,NaN,NaN,1.99
banana,3,105.0,0.4g,1mg,NaN


## Additional Resources
- [.isnull](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html) is an alias for `isna`.
- The [.value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) documentation
- [Pandas .isna documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.isna.html)

## Exercises
- Use `pd.read_csv` to read `"penguins.csv"` into a dataframe variable named `penguins`
- Write the pandas code to count the number of missing values by column
- Write the pandas necessary to get the proportion of missing values by row. Store this to a variable named `percent_missing_by_row`
- Sort the `percent_missing_by_row` series in descending order. How many of the rows are mostly empty?

In [21]:
# Use `pd.read_csv` to read `DATA_DIR / "penguins.csv"` into a dataframe variable named `penguins`


penguins = pd.read_csv(DATA_DIR / "penguins.csv")
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [22]:
# Use .isna to count the number of missing values by column


penguins.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64

In [23]:
# Write the pandas necessary to get the proportion of missing values by row. 
# Store this to a variable named `percent_missing_by_row`


percent_missing_by_row = penguins.isna().mean(axis=1)
percent_missing_by_row

0      0.000
1      0.000
2      0.000
3      0.625
4      0.000
       ...  
339    0.000
340    0.000
341    0.000
342    0.000
343    0.000
Length: 344, dtype: float64

In [24]:
# Sort the `percent_missing_by_row` series in descending order
# How many of the rows are mostly empty


percent_missing_by_row = percent_missing_by_row.sort_values(ascending=False)
mostly_empty_rows = percent_missing_by_row[percent_missing_by_row > 0.5]
print("Rows with more than half of their fields missing:", len(mostly_empty_rows))
mostly_empty_rows

Rows with more than half of their fields missing: 2


3      0.625
271    0.625
dtype: float64

---

## Course lesson: 03_Pandas_DataFrames_Continued_02.ipynb

# Intro to Pandas
by Ryan Orsinger

## Module 3: DataFrames Continued

### Pandas DataFrames Continued - Filling Missing Values
- Filling missing values
- Using `.fillna`
- Using `.loc` with DataFrames (similar to `.loc` on Series, but two-dimensional w/ rows and columns)

### Handling Missing Values is a Case in Creative Problem Solving
- There's no single right answer for all cases. 
- "It depends" is a common answer in data science. Context matters.

- Sometimes missing values might mean zero, depending on the context, so we can fill in zero.
- Sometimes, dropping entire rows or columns is appropriate
- Other times, filling missing values with the mean, the median, the mode, or a likely value is appropriate

- Sometimes, analysts drop rows with too many missing values
- Other times, analysts drop columns with too many missing values
- Missing values can also be filled with a reasonable estimation, like a median, mean, or mode value.
- Filling too many missing values can skew the original data.

In [25]:
import pandas as pd

In [26]:
# Let's generate some data with missing values. 
# Real world data often has missing values
df = pd.DataFrame([
    {
        "item": "crackers",
        "serving_size": "4 crackers",
        "calories": 10,
        "fat": "1.1g",
        "sodium": "125mg",
        "price": 2.99,
    },
    {
        "item": "club soda",
        "serving_size": "8 oz",
        "calories": None,
        "fat": None,
        "sodium": "75mg",
        "price": 2.25,

    },
    {
        "item": "apple",
        "serving_size": 2,
        "calories": 95,
        "fat": None,
        "sodium": None,
        "price": 1.99,
    },
    {
        "item": "banana",
        "serving_size": 3,
        "calories": 105,
        "fat": "0.4g",
        "sodium": "1mg",
        "price": None,
    },
    {
        "item": "spam",
        "serving_size": "1 tin",
        "calories": None,
        "fat": None,
        "sodium": None,
        "price": None,
    }
])

# Set the index to be the item name
df.set_index("item", inplace=True)
df

,serving_size,calories,fat,sodium,price
item,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99
club soda,8 oz,NaN,NaN,75mg,2.25
apple,2,95.0,NaN,NaN,1.99
banana,3,105.0,0.4g,1mg,NaN
spam,1 tin,NaN,NaN,NaN,NaN


In [27]:
# Example of filling null values with a reasonable value
# Apples and club soda don't have fat, so these missing values can be 0
df.fat = df.fat.fillna(0)
df

,serving_size,calories,fat,sodium,price
item,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99
club soda,8 oz,NaN,0,75mg,2.25
apple,2,95.0,0,NaN,1.99
banana,3,105.0,0.4g,1mg,NaN
spam,1 tin,NaN,0,NaN,NaN


### An Aside About Pandas Warnings
- Pandas warnings are not errors. The code will run. The warning is a notice, not an error that halts execution.
- Depending on your version of pandas, the above code might produce the following warning.
```
SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_index,col_indexer] = value instead
```
- Since this may impact some users, we'll move into working with `.loc` 


In [28]:
# Example of .loc's row_indexing and column_indexing
# [start_row:end_row, column_start:column_end]
# [:,] returns all rows and all columns
df.loc[:,]

,serving_size,calories,fat,sodium,price
item,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99
club soda,8 oz,NaN,0,75mg,2.25
apple,2,95.0,0,NaN,1.99
banana,3,105.0,0.4g,1mg,NaN
spam,1 tin,NaN,0,NaN,NaN


In [29]:
# Notice how we're getting the range of rows from club soda to apple
# df.loc["club soda":"banana", :]
df.loc["club soda":"banana"]

,serving_size,calories,fat,sodium,price
item,,,,,
club soda,8 oz,NaN,0,75mg,2.25
apple,2,95.0,0,NaN,1.99
banana,3,105.0,0.4g,1mg,NaN


In [30]:
# Notice how .loc uses the indexing syntax
df.loc[df.index == "apple"]

,serving_size,calories,fat,sodium,price
item,,,,,
apple,2,95.0,0,NaN,1.99


In [31]:
# Notice how .loc uses the indexing syntax
df.loc[df.serving_size == 3]

,serving_size,calories,fat,sodium,price
item,,,,,
banana,3,105.0,0.4g,1mg,NaN


In [32]:
# Notice how .loc uses the indexing syntax
df.loc[df.index == "apple", "serving_size":"fat"]

,serving_size,calories,fat
item,,,
apple,2,95.0,0


In [33]:
# All rows, show only calories as the column
df.loc[:, "calories"]

item
crackers      10.0
club soda      NaN
apple         95.0
banana       105.0
spam           NaN
Name: calories, dtype: float64

In [34]:
# Notice how : for rows returns all rows
# show all the columns from calories through price 9(inclusive)
df.loc[:, "calories":"price"]

,calories,fat,sodium,price
item,,,,
crackers,10.0,1.1g,125mg,2.99
club soda,NaN,0,75mg,2.25
apple,95.0,0,NaN,1.99
banana,105.0,0.4g,1mg,NaN
spam,NaN,0,NaN,NaN


In [35]:
# Some pandas operataions may throw a SettingWithCopyWarning
# Recommend reading the documentation carefully
# Pandas developers designed this warning because effects can be difficult to predict
# Notice how the above operation evaluated, but the warning can feel disruptive.
df.loc[df.calories.isna(), "calories"] = 0
df

,serving_size,calories,fat,sodium,price
item,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99
club soda,8 oz,0.0,0,75mg,2.25
apple,2,95.0,0,NaN,1.99
banana,3,105.0,0.4g,1mg,NaN
spam,1 tin,0.0,0,NaN,NaN


In [36]:
# An average price might be reasonable here, since we don't have other information
df.loc[df.price.isna(), "price"] = df.price.mean()
df

,serving_size,calories,fat,sodium,price
item,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99
club soda,8 oz,0.0,0,75mg,2.25
apple,2,95.0,0,NaN,1.99
banana,3,105.0,0.4g,1mg,2.41
spam,1 tin,0.0,0,NaN,2.41


In [37]:
# Actual Spam information
spam_calories = 1080
spam_fat = "96g"
spam_sodium = "4740mg" 
spam_price = 3.25

df.loc[df.index == "spam", "calories":"price"] = [spam_calories, spam_fat, spam_sodium, spam_price]
df

,serving_size,calories,fat,sodium,price
item,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99
club soda,8 oz,0.0,0,75mg,2.25
apple,2,95.0,0,NaN,1.99
banana,3,105.0,0.4g,1mg,2.41
spam,1 tin,1080.0,96g,4740mg,3.25


In [38]:
# Let's say we got in some new information about discounts
# The business manager says that we'll use discounts in the future and the existing values should be 0.
# We'll need to re-create the column and assign it zero
df["discount"] = 0

In [39]:
df

,serving_size,calories,fat,sodium,price,discount
item,,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99,0
club soda,8 oz,0.0,0,75mg,2.25,0
apple,2,95.0,0,NaN,1.99,0
banana,3,105.0,0.4g,1mg,2.41,0
spam,1 tin,1080.0,96g,4740mg,3.25,0


## Additional Resources
- Using [.fillna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html)
- [Returning-a-view-versus-a-copy in the pandas docs](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy)
- [pandas .loc documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)
- [pandas .iloc documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html)

## Exercises
- Run the cells above to remove or fill most of the missing values from the `df` variable.
- Fill the missing sodium value with a logical choice.
- Use `pd.read_csv` to read `"penguins.csv"` into a dataframe variable named `penguins`
- Fill the missing values of the `bill_length_mm` with its average
- Fill in the missing values for `bill_depth_mm` with its average
- Fill in the missing values for `body_mass_g` with its average
- Run `.value_counts` on the `sex` column
- Fill the missing values in the `sex` column with the `mode` (Follow .mode() with [0] to access the string value)
- Run `.value_counts` on the `sex` column again, after filling the missing values

In [40]:
# Fill the missing sodium value with a logical choice.


df.loc[df.sodium.isna(), "sodium"] = "0mg"
df

,serving_size,calories,fat,sodium,price,discount
item,,,,,,
crackers,4 crackers,10.0,1.1g,125mg,2.99,0
club soda,8 oz,0.0,0,75mg,2.25,0
apple,2,95.0,0,0mg,1.99,0
banana,3,105.0,0.4g,1mg,2.41,0
spam,1 tin,1080.0,96g,4740mg,3.25,0


In [41]:
# df.loc[row_indexer, column_indexer] = value


In [42]:
# Use `pd.read_csv` to read `DATA_DIR / "penguins.csv"` into a dataframe variable named `penguins`


penguins = pd.read_csv(DATA_DIR / "penguins.csv")
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [43]:
# Fill the missing values of the `bill_length_mm` with its average


penguins["bill_length_mm"] = penguins["bill_length_mm"].fillna(penguins["bill_length_mm"].mean())
penguins["bill_length_mm"].isna().sum()

np.int64(0)

In [44]:
# Fill in the missing values for `bill_depth_mm` with its average


penguins["bill_depth_mm"] = penguins["bill_depth_mm"].fillna(penguins["bill_depth_mm"].mean())
penguins["bill_depth_mm"].isna().sum()

np.int64(0)

In [45]:
# Fill in the missing values for `body_mass_g` with its average


penguins["body_mass_g"] = penguins["body_mass_g"].fillna(penguins["body_mass_g"].mean())
penguins["body_mass_g"].isna().sum()

np.int64(0)

In [46]:
# Run `.value_counts` on the `sex` column


penguins["sex"].value_counts(dropna=False)

sex
male      168
female    165
NaN        11
Name: count, dtype: int64

In [47]:
# Fill the missing values in the `sex` column with the `mode` (Follow .mode() with [0] to access the string value)


penguins["sex"] = penguins["sex"].fillna(penguins["sex"].mode()[0])
penguins["sex"].isna().sum()

np.int64(0)

In [48]:
# Run `.value_counts` on the `sex` column again, after filling the missing values


penguins["sex"].value_counts()

sex
male      179
female    165
Name: count, dtype: int64